# Algorithmic Trading and Quantitative Strategies
## Part 8: Factor Investing — Theory and Python Implementation
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. Foundations of Factor Investing
2. Multi-Factor Models
3. Factor Construction in Python
4. Factor Mimicking Portfolios
5. Information Coefficient Analysis
6. Performance Evaluation
7. Application: Turkish Equities
8. Exercises

## 1. Foundations of Factor Investing

### 1.1 From CAPM to Multi-Factor Models

The **Capital Asset Pricing Model (CAPM)** states that the expected excess return of an asset is proportional to its market beta:

$$E[R_i] - R_f = \beta_i (E[R_m] - R_f)$$

where:
- $R_i$ = return of asset $i$
- $R_f$ = risk-free rate
- $R_m$ = market return
- $\beta_i = \frac{\text{Cov}(R_i, R_m)}{\text{Var}(R_m)}$

However, CAPM fails to explain many observed return patterns. This led to **multi-factor models** that include additional risk factors.

### 1.2 Types of Factors

**Macroeconomic factors:**
- GDP growth, inflation, interest rates, credit spreads

**Fundamental (style) factors:**
- Value (P/E, P/B), Size (market cap), Momentum, Quality (ROE, profitability), Low volatility

**Statistical factors:**
- Derived from PCA or factor analysis of return covariance matrix

### 1.3 Why Factor Investing Works

**Risk-based explanations:**
- Value stocks are riskier (distress risk) → higher expected returns as compensation
- Small stocks are less liquid → liquidity premium

**Behavioral explanations:**
- Overreaction/underreaction to news
- Herding behavior
- Disposition effect
- Anchoring to past prices/earnings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)

## 2. Multi-Factor Models

### 2.1 Fama-French Three-Factor Model

Fama and French (1993) added two factors to CAPM:

$$R_i - R_f = \alpha_i + \beta_{i,MKT} (R_m - R_f) + \beta_{i,SMB} \cdot SMB + \beta_{i,HML} \cdot HML + \epsilon_i$$

where:
- **SMB** (Small Minus Big): Return spread between small-cap and large-cap stocks
- **HML** (High Minus Low): Return spread between value (high B/M) and growth (low B/M) stocks

### 2.2 Fama-French Five-Factor Model

Fama and French (2015) added:

$$R_i - R_f = \alpha_i + \beta_{MKT} MKT + \beta_{SMB} SMB + \beta_{HML} HML + \beta_{RMW} RMW + \beta_{CMA} CMA + \epsilon_i$$

- **RMW** (Robust Minus Weak): Profitability factor
- **CMA** (Conservative Minus Aggressive): Investment factor

### 2.3 Other Common Factors

| Factor | Description | Long | Short |
|:---|:---|:---|:---|
| Momentum | Past winners outperform losers | 12-1 month winners | 12-1 month losers |
| Low Vol | Low-risk stocks outperform | Low beta/vol | High beta/vol |
| Quality | Profitable firms outperform | High ROE/margins | Low ROE/margins |
| Dividend | High yield outperforms | High dividend | Low dividend |

In [ ]:
import getFamaFrenchFactors as gff
import matplotlib.pyplot as plt

# Fetch Fama-French 5 factors (monthly)
ff5 = gff.famaFrench5Factor(frequency="m")
ff5.set_index("date_ff_factors", inplace=True)

# Fetch Momentum separately
mom = gff.momentumFactor(frequency="m")
mom.set_index("date_ff_factors", inplace=True)

# Combine all 6 factors
df = ff5.join(mom[["MOM"]])

# Plot cumulative returns
factors = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "MOM"]
cumulative = (1 + df[factors]).cumprod()

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
for ax, factor in zip(axes.flat, factors):
    ax.plot(cumulative.index, cumulative[factor], linewidth=1)
    ax.set_title(factor, fontsize=13)
    ax.grid(True, alpha=0.3)

fig.suptitle("Fama-French Factors: Cumulative Returns", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 3. Factor Construction in Python

### 3.1 Data Preparation

Before constructing factors, raw data needs to be cleaned:

1. **Outlier treatment:** Winsorization (cap extreme values at percentiles)
2. **Missing data:** Forward-fill or drop
3. **Transformations:** Log, rank, standardize

In [ ]:
def winsorize(series, limits=(0.01, 0.99)):
    """Winsorize a series at given percentiles."""
    lower = series.quantile(limits[0])
    upper = series.quantile(limits[1])
    return series.clip(lower=lower, upper=upper)

def zscore(series):
    """Standardize to z-score."""
    return (series - series.mean()) / series.std()

def rank_normalize(series):
    """Rank-based normalization to uniform [0, 1]."""
    return series.rank(pct=True)

# Example: simulate cross-sectional data
np.random.seed(42)
n_stocks = 50

stock_data = pd.DataFrame({
    'ticker': [f'STOCK_{i:02d}' for i in range(n_stocks)],
    'market_cap': np.random.lognormal(mean=8, sigma=1.5, size=n_stocks),
    'pe_ratio': np.random.lognormal(mean=2.5, sigma=0.6, size=n_stocks),
    'roe': np.random.normal(0.15, 0.10, size=n_stocks),
    'momentum_12m': np.random.normal(0.10, 0.30, size=n_stocks),
    'volatility': np.random.lognormal(mean=-1.5, sigma=0.5, size=n_stocks),
})
stock_data = stock_data.set_index('ticker')

# Clean data
print("Before winsorization:")
print(stock_data.describe().round(3))

for col in ['pe_ratio', 'roe', 'momentum_12m']:
    stock_data[col] = winsorize(stock_data[col])

# Add z-scores
for col in ['pe_ratio', 'roe', 'momentum_12m']:
    stock_data[f'{col}_z'] = zscore(stock_data[col])

print("\nAfter winsorization + z-score:")
print(stock_data.filter(like='_z').describe().round(3))

### 3.2 Composite Factor Scores

Combine multiple signals into a composite score for each factor:

$$\text{Score}_i = \sum_k w_k \cdot z_{i,k}$$

where $z_{i,k}$ is the z-score of signal $k$ for stock $i$.

In [ ]:
# Construct composite factor scores
# Value factor: low P/E is good → negate
stock_data['value_score'] = -stock_data['pe_ratio_z']

# Quality factor: high ROE is good
stock_data['quality_score'] = stock_data['roe_z']

# Momentum factor: high past return is good
stock_data['momentum_score'] = stock_data['momentum_12m_z']

# Combined alpha score (equal weight)
stock_data['alpha_score'] = (
    stock_data['value_score'] + 
    stock_data['quality_score'] + 
    stock_data['momentum_score']
) / 3

# Rank stocks
stock_data['alpha_rank'] = stock_data['alpha_score'].rank(ascending=False)

print("Top 10 stocks by Alpha Score:")
print(stock_data.nlargest(10, 'alpha_score')[
    ['market_cap', 'pe_ratio', 'roe', 'momentum_12m', 'alpha_score', 'alpha_rank']
].round(3))

## 4. Factor Mimicking Portfolios (FMP)

A **Factor Mimicking Portfolio** is a long-short portfolio designed to capture the return of a specific factor.

### 4.1 Portfolio Sort Approach (Quintile Portfolios)

1. Sort stocks by factor score
2. Divide into quintiles (5 groups)
3. Long the top quintile, short the bottom quintile
4. The spread (Q5 - Q1) is the factor return

In [ ]:
# Simulate monthly cross-sectional factor portfolio construction
np.random.seed(42)
n_stocks = 100
n_months = 60
n_quintiles = 5

# Simulate monthly stock returns and factor scores
monthly_factor_returns = []

for month in range(n_months):
    # Factor scores (value signal)
    factor_score = np.random.normal(0, 1, n_stocks)
    
    # Returns have a small relationship with factor score + noise
    alpha = 0.003  # Monthly alpha per unit of factor exposure
    returns = alpha * factor_score + np.random.normal(0, 0.08, n_stocks)
    
    # Sort into quintiles
    df_month = pd.DataFrame({
        'score': factor_score,
        'return': returns,
    })
    df_month['quintile'] = pd.qcut(df_month['score'], n_quintiles, labels=range(1, n_quintiles+1))
    
    # Quintile average returns
    q_returns = df_month.groupby('quintile')['return'].mean()
    
    # Factor return = Q5 - Q1 (long-short)
    factor_ret = q_returns.iloc[-1] - q_returns.iloc[0]
    monthly_factor_returns.append({
        'month': month,
        'Q1': q_returns.iloc[0],
        'Q2': q_returns.iloc[1],
        'Q3': q_returns.iloc[2],
        'Q4': q_returns.iloc[3],
        'Q5': q_returns.iloc[4],
        'L/S': factor_ret,
    })

fmp_df = pd.DataFrame(monthly_factor_returns)

# Quintile performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of average returns by quintile
avg_returns = fmp_df[['Q1','Q2','Q3','Q4','Q5']].mean() * 12  # Annualize
axes[0].bar(range(5), avg_returns, color=['red','salmon','grey','lightgreen','green'])
axes[0].set_xticks(range(5))
axes[0].set_xticklabels(['Q1\n(Low)', 'Q2', 'Q3', 'Q4', 'Q5\n(High)'])
axes[0].set_ylabel('Annualized Return')
axes[0].set_title('Factor Quintile Returns')
axes[0].axhline(y=0, color='k', linewidth=0.5)

# Cumulative L/S return
cum_ls = (1 + fmp_df['L/S']).cumprod()
axes[1].plot(cum_ls, 'b-', linewidth=2)
axes[1].set_title('Cumulative Long-Short Factor Return')
axes[1].set_ylabel('Growth of $1')
axes[1].set_xlabel('Month')
axes[1].axhline(y=1, color='k', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

# Statistics
ls_returns = fmp_df['L/S']
t_stat, p_value = stats.ttest_1samp(ls_returns, 0)
print(f"Factor Return Statistics:")
print(f"  Ann. Return:    {ls_returns.mean() * 12:.2%}")
print(f"  Ann. Volatility:{ls_returns.std() * np.sqrt(12):.2%}")
print(f"  Sharpe Ratio:   {ls_returns.mean() / ls_returns.std() * np.sqrt(12):.3f}")
print(f"  t-statistic:    {t_stat:.3f}")
print(f"  p-value:        {p_value:.4f}")

### 4.2 Cross-Sectional Regression (Fama-MacBeth)

The Fama-MacBeth (1973) method:

1. **Each period:** Run a cross-sectional regression of returns on factor exposures
$$R_{i,t} = \gamma_{0,t} + \gamma_{1,t} X_{i,t-1} + \epsilon_{i,t}$$

2. **Across time:** The factor premium is the time-series average of $\gamma_{1,t}$
$$\hat{\lambda} = \frac{1}{T} \sum_{t=1}^{T} \hat{\gamma}_{1,t}$$

3. **Test significance:** Using the standard error of $\hat{\gamma}_{1,t}$ across time

In [ ]:
def fama_macbeth(returns_panel, factors_panel):
    """
    Fama-MacBeth cross-sectional regression.
    
    Parameters:
    -----------
    returns_panel : pd.DataFrame (T x N) - Asset returns
    factors_panel : dict of pd.DataFrame (T x N) - Factor exposures
    
    Returns:
    --------
    dict with factor premia estimates and t-statistics
    """
    T = len(returns_panel)
    factor_names = list(factors_panel.keys())
    gammas = {f: [] for f in factor_names}
    gammas['intercept'] = []
    
    for t in range(T):
        ret = returns_panel.iloc[t].dropna()
        
        X = pd.DataFrame({f: factors_panel[f].iloc[t] for f in factor_names})
        X = X.loc[ret.index].dropna()
        ret = ret.loc[X.index]
        
        if len(ret) < 10:
            continue
        
        X_const = sm.add_constant(X)
        model = sm.OLS(ret, X_const).fit()
        
        gammas['intercept'].append(model.params['const'])
        for f in factor_names:
            gammas[f].append(model.params[f])
    
    # Compute statistics
    results = {}
    for name in ['intercept'] + factor_names:
        g = np.array(gammas[name])
        mean_g = g.mean()
        se_g = g.std() / np.sqrt(len(g))
        t_stat = mean_g / se_g
        results[name] = {
            'premium': mean_g,
            'std_error': se_g,
            't_stat': t_stat,
            'p_value': 2 * (1 - stats.t.cdf(abs(t_stat), len(g)-1)),
        }
    
    return pd.DataFrame(results).T

# Simulate cross-sectional data for FM regression
np.random.seed(42)
n_stocks, n_months = 50, 60
dates = pd.date_range('2019-01-01', periods=n_months, freq='ME')
tickers = [f'S{i:02d}' for i in range(n_stocks)]

# Factor exposures (lagged)
value_exp = pd.DataFrame(np.random.normal(0, 1, (n_months, n_stocks)), 
                         index=dates, columns=tickers)
mom_exp = pd.DataFrame(np.random.normal(0, 1, (n_months, n_stocks)),
                       index=dates, columns=tickers)

# Returns = alpha + factor_premium * exposure + noise
returns_panel = (
    0.001 +
    0.003 * value_exp + 
    0.005 * mom_exp + 
    pd.DataFrame(np.random.normal(0, 0.08, (n_months, n_stocks)),
                 index=dates, columns=tickers)
)

# Run Fama-MacBeth
fm_results = fama_macbeth(returns_panel, {'value': value_exp, 'momentum': mom_exp})
print("Fama-MacBeth Results:")
print(fm_results.round(4))

### 4.3 Statistical Factor Approach (PCA)

Principal Component Analysis (PCA) extracts latent factors from the covariance matrix of returns.

The first few principal components capture the dominant return patterns (market, sector, style).

In [ ]:
# PCA on stock returns
returns_matrix = returns_panel.values
scaler = StandardScaler()
returns_scaled = scaler.fit_transform(returns_matrix)

pca = PCA(n_components=10)
pca.fit(returns_scaled)

# Variance explained
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 11), pca.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('PCA: Variance Explained by Component')

cum_var = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, 11), cum_var, 'bo-')
axes[1].axhline(y=0.8, color='r', linestyle='--', label='80% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('PCA: Cumulative Variance Explained')
axes[1].legend()

plt.tight_layout()
plt.show()

# Factor returns (PC scores)
pc_returns = pca.transform(returns_scaled)[:, :3]
pc_df = pd.DataFrame(pc_returns, index=dates, columns=['PC1', 'PC2', 'PC3'])

print(f"\nFirst 3 PCs explain {cum_var[2]:.1%} of variance")
print(f"\nPC Factor Return Statistics:")
print(pc_df.describe().round(4))

## 5. Information Coefficient Analysis

The **Information Coefficient (IC)** measures the predictive power of a factor signal. It is the cross-sectional correlation between the factor signal and subsequent returns:

$$IC_t = \text{Corr}(\text{Signal}_{t-1}, R_t)$$

- **IC > 0:** Factor has predictive power
- **IC ~ 0.05:** Typical for good factors
- **IC > 0.1:** Exceptional
- **IR (Information Ratio):** $IR = \frac{\overline{IC}}{\sigma(IC)}$ — IC adjusted for consistency

In [ ]:
# Compute IC time series
np.random.seed(42)
n_stocks, n_months = 100, 120

ic_series = []
for t in range(n_months):
    signal = np.random.normal(0, 1, n_stocks)
    # Returns weakly correlated with signal
    returns = 0.003 * signal + np.random.normal(0, 0.08, n_stocks)
    ic = np.corrcoef(signal, returns)[0, 1]
    ic_series.append(ic)

ic_series = pd.Series(ic_series, index=pd.date_range('2015-01-01', periods=n_months, freq='ME'))

print(f"IC Analysis:")
print(f"  Mean IC:    {ic_series.mean():.4f}")
print(f"  Std IC:     {ic_series.std():.4f}")
print(f"  IC > 0:     {(ic_series > 0).mean():.1%}")
print(f"  IR:         {ic_series.mean() / ic_series.std():.4f}")

# t-test for IC significance
t_stat, p_val = stats.ttest_1samp(ic_series, 0)
print(f"  t-stat:     {t_stat:.4f}")
print(f"  p-value:    {p_val:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(len(ic_series)), ic_series, 
            color=['green' if x > 0 else 'red' for x in ic_series], alpha=0.7)
axes[0].axhline(y=ic_series.mean(), color='blue', linestyle='--', label=f'Mean IC = {ic_series.mean():.3f}')
axes[0].set_title('Monthly Information Coefficient')
axes[0].legend()

axes[1].hist(ic_series, bins=20, edgecolor='black', alpha=0.7)
axes[1].axvline(x=ic_series.mean(), color='red', linestyle='--')
axes[1].set_title('IC Distribution')
axes[1].set_xlabel('IC')

plt.tight_layout()
plt.show()

## 6. Performance Evaluation of Factor Portfolios

Key metrics for evaluating factor portfolios:

In [ ]:
def evaluate_factor_portfolio(returns, name="Factor"):
    """Comprehensive evaluation of a return series."""
    ann_ret = returns.mean() * 12
    ann_vol = returns.std() * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    
    # Drawdown
    cum = (1 + returns).cumprod()
    dd = cum / cum.cummax() - 1
    max_dd = dd.min()
    
    # Calmar ratio
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0
    
    # t-test
    t_stat, p_val = stats.ttest_1samp(returns, 0)
    
    # Skewness and kurtosis
    skew = returns.skew()
    kurt = returns.kurtosis()
    
    print(f"{'='*40}")
    print(f"{name} Portfolio Performance")
    print(f"{'='*40}")
    print(f"Ann. Return:    {ann_ret:.2%}")
    print(f"Ann. Volatility:{ann_vol:.2%}")
    print(f"Sharpe Ratio:   {sharpe:.3f}")
    print(f"Max Drawdown:   {max_dd:.2%}")
    print(f"Calmar Ratio:   {calmar:.3f}")
    print(f"Skewness:       {skew:.3f}")
    print(f"Excess Kurtosis:{kurt:.3f}")
    print(f"t-statistic:    {t_stat:.3f}")
    print(f"p-value:        {p_val:.4f}")
    print(f"% Positive:     {(returns > 0).mean():.1%}")
    
    return {'ann_ret': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe, 
            'max_dd': max_dd, 't_stat': t_stat}

# Evaluate the simulated factor portfolio
evaluate_factor_portfolio(fmp_df['L/S'], "Value Factor L/S")

## 7. Application: Factor Investing with Real Data

Let us construct a simple momentum factor using real stock data.

In [ ]:
# Download data for a basket of stocks
tickers = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'META', 'NVDA', 'TSLA', 'JPM', 'JNJ', 'V',
           'PG', 'UNH', 'HD', 'MA', 'DIS', 'NFLX', 'ADBE', 'CRM', 'PEP', 'KO']

prices = yf.download(tickers, start='2018-01-01', end='2025-12-31')['Close']
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.droplevel(1)
returns = prices.pct_change().dropna()

print(f"Data: {len(returns)} days, {len(returns.columns)} stocks")
print(f"Date range: {returns.index[0].date()} to {returns.index[-1].date()}")

In [ ]:
# Monthly momentum factor construction
monthly_prices = prices.resample('ME').last()
monthly_returns = monthly_prices.pct_change()

# 12-1 month momentum (skip most recent month)
momentum_signal = monthly_prices.pct_change(11).shift(1)

# Construct monthly L/S portfolio
factor_returns = []

for i in range(12, len(monthly_returns)):
    signal = momentum_signal.iloc[i].dropna()
    ret = monthly_returns.iloc[i].dropna()
    
    common = signal.index.intersection(ret.index)
    if len(common) < 6:
        continue
    
    signal = signal[common]
    ret = ret[common]
    
    # Long top quintile, short bottom quintile
    n = len(common)
    top = signal.nlargest(n // 5).index
    bottom = signal.nsmallest(n // 5).index
    
    long_ret = ret[top].mean()
    short_ret = ret[bottom].mean()
    ls_ret = long_ret - short_ret
    
    factor_returns.append({
        'date': monthly_returns.index[i],
        'long': long_ret,
        'short': short_ret,
        'L/S': ls_ret,
    })

mom_df = pd.DataFrame(factor_returns).set_index('date')

# Evaluate
evaluate_factor_portfolio(mom_df['L/S'], "Momentum Factor (12-1 Month)")

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
cum = (1 + mom_df[['long', 'short', 'L/S']]).cumprod()
for col in cum.columns:
    axes[0].plot(cum[col], label=col, linewidth=1.5)
axes[0].set_title('Momentum Factor: Cumulative Returns')
axes[0].legend()
axes[0].axhline(y=1, color='k', linestyle='--', linewidth=0.5)
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(mom_df)), mom_df['L/S'],
           color=['green' if x > 0 else 'red' for x in mom_df['L/S']], alpha=0.7)
axes[1].set_title('Monthly L/S Returns')
axes[1].set_ylabel('Return')

plt.tight_layout()
plt.show()

## 8. Exercises

1. **Value Factor**: Using the 20 US stocks above, construct a value factor using the trailing P/E ratio (use `yf.Ticker(symbol).info` to get the P/E ratio). Evaluate the long-short portfolio.

2. **Low Volatility Factor**: Sort stocks by their trailing 60-day volatility. Construct a long-short portfolio (long low-vol, short high-vol). Does the low volatility anomaly appear?

3. **Multi-Factor Model**: Combine momentum and value scores into a composite score. Compare the performance of the composite factor vs. individual factors.

4. **Fama-MacBeth Regression**: For the 20 stocks, run monthly Fama-MacBeth regressions with momentum signal and log market cap as factors. Are the factor premia significant?

---
### References
- Fama, E. F., & French, K. R. (1993). *Common Risk Factors in the Returns on Stocks and Bonds.* Journal of Financial Economics.
- Fama, E. F., & French, K. R. (2015). *A Five-Factor Asset Pricing Model.* Journal of Financial Economics.
- Fama, E. F., & MacBeth, J. D. (1973). *Risk, Return, and Equilibrium: Empirical Tests.* Journal of Political Economy.
- Grinold, R. & Kahn, R. (2000). *Active Portfolio Management.* McGraw-Hill.
- Fabozzi, Focardi, & Kolm (2010). *Quantitative Equity Investing.* Wiley.
- Bali, Engle & Murray (2016). *Empirical Asset Pricing: The Cross Section of Stock Returns.* Wiley.